In [2]:
!pip install ta

  Preparing metadata (setup.py) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=b1539b09bafadef57aec92bbd5f1f5f8b436844c62013c3d9161828380982441
  Stored in directory: /root/.cache/pip/wheels/5c/a1/5f/c6b85a7d9452057be4ce68a8e45d77ba34234a6d46581777c6
Successfully built ta


In [3]:
!pip install openai

In [10]:
import yfinance
import pandas
import ta

In [23]:
from crewai import Agent
#currently adding manually Stock name
stockName = "HASCOL.KA"

In [29]:
#checking the stock current data
import yfinance as yf

stock = yf.Ticker(stockName)

info = stock.info

print(info["longName"])
print(info.get("currentPrice"))
print(info["marketCap"])

Hascol Petroleum Limited
None
6074653696


In [25]:
# add technical analysis to get the 6month data


df = yf.download(stockName, period="6mo")

close_prices = df["Close"].squeeze()
df["RSI"] = ta.momentum.RSIIndicator(close_prices).rsi()


print(df)

/tmp/ipykernel_3027/3945228238.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(stockName, period="6mo")
[*********************100%***********************]  1 of 1 completed

Price           Close       High        Low       Open     Volume        RSI
Ticker      HASCOL.KA  HASCOL.KA  HASCOL.KA  HASCOL.KA  HASCOL.KA           
Date                                                                        
2025-11-14  14.550000  14.850000  13.620000  14.000000   19781976        NaN
2025-11-17  14.580000  14.950000  14.500000  14.950000   10065613        NaN
2025-11-18  14.550000  14.850000  14.440000  14.700000   10806079        NaN
2025-11-19  14.320000  14.780000  14.300000  14.600000    5683402        NaN
2025-11-20  14.460000  14.800000  14.300000  14.600000   11378348        NaN
...               ...        ...        ...        ...        ...        ...
2026-05-07  23.250000  23.600000  22.400000  22.879999   44741876  75.027508
2026-05-08  24.559999  25.389999  22.879999  23.150000  100747421  78.867384
2026-05-11  24.730000  25.340000  24.280001  24.330000   35395971  79.311951
2026-05-12  22.320000  25.150000  22.260000  25.000000  113570988  60.031538

In [26]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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",
)

In [44]:
# Convert columns to 1D
high = df["High"].squeeze()
low = df["Low"].squeeze()
close = df["Close"].squeeze()
volume = df["Volume"].squeeze()

# RSI
df["RSI"] = ta.momentum.RSIIndicator(close).rsi()

# MACD
macd = ta.trend.MACD(close)

df["MACD"] = macd.macd()

# SMA
df["SMA20"] = ta.trend.sma_indicator(close, window=20)

# EMA
df["EMA20"] = ta.trend.ema_indicator(close, window=20)

# Bollinger Bands
bb = ta.volatility.BollingerBands(close)

df["BB_HIGH"] = bb.bollinger_hband()
df["BB_LOW"] = bb.bollinger_lband()

# Stochastic
stoch = ta.momentum.StochasticOscillator(
    high=high,
    low=low,
    close=close
)

df["Stoch"] = stoch.stoch()

# ADX
adx = ta.trend.ADXIndicator(
    high=high,
    low=low,
    close=close
)

df["ADX"] = adx.adx()

# Print latest indicators
print(df[[
    "Close",
    "RSI",
    "MACD",
    "SMA20",
    "EMA20",
    "BB_HIGH",
    "BB_LOW",
    "Stoch",
    "ADX"
]].tail())


Price           Close        RSI      MACD    SMA20      EMA20    BB_HIGH  \
Ticker      HASCOL.KA                                                       
Date                                                                        
2026-05-07  23.250000  75.027508  1.269151  19.6370  20.099860  23.284235   
2026-05-08  24.559999  78.867384  1.462102  19.9950  20.524635  24.073728   
2026-05-11  24.730000  79.311951  1.610174  20.3440  20.925146  24.773995   
2026-05-12  22.320000  60.031538  1.515585  20.6295  21.057989  24.787766   
2026-05-13  22.209999  59.322677  1.415430  20.8875  21.167704  24.755568   

Price          BB_LOW      Stoch        ADX  
Ticker                                       
Date                                         
2026-05-07  15.989765  92.281893  27.096306  
2026-05-08  15.916272  89.136126  29.754303  
2026-05-11  15.914005  90.807802  32.222444  
2026-05-12  16.471234  57.242346  32.231038  
2026-05-13  17.019432  46.911512  31.342657  


In [46]:
prompt = f"""
Stock: HASCOL
Price: 22.32

Indicators:
RSI: {df["RSI"].iloc[-1]}
MACD: {df["MACD"].iloc[-1]}
SMA20: {df["SMA20"].iloc[-1]}
ADX: {df["ADX"].iloc[-1]}

Should investor buy this stock?
give the answer should i buy or not
Explain reasons.
"""

response = client.chat.completions.create(
    model="deepseek/deepseek-v4-flash:free",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)
print(response.choices[0].message.content)

Based on the provided indicators, the investor should **buy** the stock.

### Reasons:
- **RSI (59.32)** – The Relative Strength Index is above 50 but below 70, indicating bullish momentum without being overbought, leaving room for further upside.
- **MACD (1.415)** – A positive MACD value above the signal line (not given but implied) suggests strong upward momentum and bullish trend.
- **SMA20 (20.89)** – The current price (22.32) is above the 20-day simple moving average, confirming a short-term uptrend.
- **ADX (31.34)** – A value above 25 indicates a strong trend. Combined with the other bullish indicators, it supports a sustained upward move.

Overall, the technical signals align positively, making this a favorable buying opportunity.


In [ ]:
!pip install crewai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 5.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [14]:
from crewai import Agent
market_agent = Agent(
    role="Market Analyst",
    goal="Analyze stock fundamentals",
    backstory="Expert financial analyst"
)